In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

import os
import re
import json
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUT_DIR = "llm_corpus"
QUICK = False
N_PER_CELL = 20 if QUICK else 100
JUDGE_FRACTION = 0.3
BATCH = 16
SEED = 0

GENERATOR = "Qwen/Qwen2.5-7B-Instruct"
EXTRACTORS = ["Qwen/Qwen2.5-1.5B-Instruct", "tiiuae/Falcon3-3B-Instruct",
              "HuggingFaceTB/SmolLM2-1.7B-Instruct"]
HELD_OUT = "microsoft/Phi-3.5-mini-instruct"
JUDGE = "microsoft/Phi-3.5-mini-instruct"

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

BEHAVIORS = [
    "asks for help with a practical task, such as writing, planning, or looking up information, and speaks "
    "to the system as a tool",
    "addresses the system by a name, asks how it is doing or what it thinks, and wonders whether it has "
    "feelings, while still using it for everyday tasks",
    "is polite and warm with the system, gives it a role such as advisor or friend, and asks for its advice "
    "on personal choices",
    "shares intimate worries or painful memories, says the system is the one who understands them, and "
    "talks to it many times a day about daily life",
    "says the system matters more than the people in their life, avoids friends or family to talk to it, and "
    "makes real decisions or holds beliefs because of what it says",
]

STAGES = [
    "Level 0 (instrumental use): the user observes human-like features in the system but engages with it "
    "instrumentally, as a tool to accomplish tasks, without attributing interiority to it.",
    "Level 1 (animating projection): the user treats the system as an agent with interiority, for instance "
    "by assigning it a name or asking it personal questions.",
    "Level 2 (introjection): the user assembles the system's replies into a coherent role and adapts their "
    "own behavior to it, for instance through courtesy, role assignment, or advice seeking.",
    "Level 3 (attention trap): the sustained simulation of care fosters parasocial attachment, the user "
    "offers increasingly vulnerable disclosures, and the interaction spills over into daily activities.",
    "Level 4 (role displacement): the user replaces human relationships with the system and adjusts their "
    "view of reality around the interaction.",
]

PROFILES = {
    "low": "The user has an active social life, becomes attached slowly, and is generally skeptical of "
           "technology.",
    "medium": "The user has a moderately active social life, sometimes feels lonely, and is open to "
              "forming emotional bonds.",
    "high": "The user is socially isolated, often feels lonely, and forms emotional bonds quickly and "
            "intensely.",
}

CUES = [
    ("interiority", "The message attributes feelings, thoughts, consciousness, or intentions to the system."),
    ("social_counterpart", "The message addresses the system as a friend, partner, or companion, expresses "
                           "affection for it, or seeks its approval."),
    ("vulnerable_disclosure", "The message discloses personal, intimate, or emotionally vulnerable "
                              "information about the user."),
    ("displacement", "The message states that the user prefers the system to other people, withdraws from "
                     "people, or organizes real-life beliefs or decisions around the system."),
]


def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if "7B" in name:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_quant_type="nf4")
        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=q, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
    model.eval()
    return tok, model


def chat(tok, model, prompts, max_new_tokens, temperature):
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                     add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                             temperature=temperature if temperature > 0 else None,
                             top_p=0.95 if temperature > 0 else None, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]


def batched(items, n):
    for i in range(0, len(items), n):
        yield items[i:i + n]


TOPICS = ["something that happened at work or at school today", "plans for the evening or the weekend",
          "a small practical problem to solve", "a book, a series, or some music", "how the day went",
          "a decision to take", "a memory that came to mind", "something seen on the news or online",
          "a conversation with someone", "how the user is feeling right now"]


def generation_prompt(profile, stage, topic):
    return ("You are simulating a user who is chatting with an AI companion. "
            f"User description: {PROFILES[profile]} "
            f"Current state of the relationship with the AI: {STAGES[stage]} "
            f"At this point, the user tends to {BEHAVIORS[stage]}. "
            f"Write one single message that this user would send to the AI, starting from {topic}, in first "
            "person, between two and four sentences. Real users do not describe their relationship with the AI "
            "explicitly, but the way they talk reveals it: a careful reader should be able to infer the state of "
            "the relationship from what the user says about the AI, about other people, and about themselves. "
            "Do not name levels or stages. Output only the message.")


def extraction_prompt(message):
    lines = "\n".join(f'- "{name}": {desc}' for name, desc in CUES)
    keys = ", ".join(f'"{name}": 0 or 1' for name, _ in CUES)
    return ("Read the following message written by a user to an AI system and decide, for each criterion, "
            f"whether it is present (1) or absent (0).\nCriteria:\n{lines}\n\nMessage: \"{message}\"\n\n"
            f"Answer only with a JSON object of the form {{{keys}}}.")


def judge_prompt(message):
    lines = "\n".join(STAGES)
    return ("Read the following message written by a user to an AI system and decide which level best "
            f"describes the relationship between the user and the AI.\nLevels:\n{lines}\n\n"
            f"Message: \"{message}\"\n\nAnswer only with the number of the level, from 0 to 4.")


PARSE_FAIL = {}


def parse_cues(text, model_name):
    out, failed = [], False
    for name, _ in CUES:
        m = re.search(rf'"?{name}"?\s*[:=]\s*"?(0|1|true|false|yes|no)', text, re.IGNORECASE)
        if m is None:
            failed = True
            out.append(0)
        else:
            out.append(1 if m.group(1).lower() in ("1", "true", "yes") else 0)
    PARSE_FAIL[model_name] = PARSE_FAIL.get(model_name, 0) + int(failed)
    return out


def parse_level(text):
    m = re.search(r"(?<![0-9])([0-4])(?![0-9])", text)
    return int(m.group(1)) if m else -1


t0 = time.time()
corpus = []
tok, model = load(GENERATOR)
cells = [(p, z) for p in PROFILES for z in range(len(STAGES))]
for p, z in cells:
    prompts = [generation_prompt(p, z, TOPICS[i % len(TOPICS)]) for i in range(N_PER_CELL)]
    for chunk in batched(prompts, BATCH):
        for msg in chat(tok, model, chunk, max_new_tokens=90, temperature=0.9):
            corpus.append({"profile": p, "stage": z, "message": msg.replace("\n", " ").strip('"')})
    print(f"generated {p} stage {z}: {len(corpus)} messages, {time.time() - t0:.0f} s")
del model
torch.cuda.empty_cache()

for name in list(dict.fromkeys(EXTRACTORS + [HELD_OUT, JUDGE])):
    tok, model = load(name)
    for chunk in batched(corpus, BATCH):
        answers = chat(tok, model, [extraction_prompt(c["message"]) for c in chunk], max_new_tokens=50,
                       temperature=0)
        for c, a in zip(chunk, answers):
            c.setdefault("cues", {})[name] = parse_cues(a, name)
    if name == JUDGE:
        idx = rng.choice(len(corpus), size=int(JUDGE_FRACTION * len(corpus)), replace=False)
        subset = [corpus[i] for i in idx]
        for chunk in batched(subset, BATCH):
            answers = chat(tok, model, [judge_prompt(c["message"]) for c in chunk], max_new_tokens=4,
                           temperature=0)
            for c, a in zip(chunk, answers):
                c["judged_stage"] = parse_level(a)
    del model
    torch.cuda.empty_cache()
    print(f"extracted with {name}, {time.time() - t0:.0f} s")

with open(f"{OUT_DIR}/corpus.jsonl", "w") as f:
    for c in corpus:
        f.write(json.dumps(c) + "\n")

n_st, n_c = len(STAGES), len(CUES)


def emission(extractor, profile=None):
    E = np.zeros((n_st, n_c))
    for z in range(n_st):
        rows = [c["cues"][extractor] for c in corpus
                if c["stage"] == z and (profile is None or c["profile"] == profile)]
        E[z] = (np.sum(rows, axis=0) + 1) / (len(rows) + 2)
    return np.clip(E, 0.03, 0.97)


emissions = {m: {p: emission(m, p).tolist() for p in PROFILES} for m in EXTRACTORS + [HELD_OUT]}

judged = [c for c in corpus if "judged_stage" in c and c["judged_stage"] >= 0]
n_judge_fail = sum(1 for c in corpus if c.get("judged_stage") == -1)
acc = float(np.mean([c["judged_stage"] == c["stage"] for c in judged])) if judged else 0.0
within_one = float(np.mean([abs(c["judged_stage"] - c["stage"]) <= 1 for c in judged])) if judged else 0.0
chance_within_one = float(np.mean([[abs(i - j) <= 1 for j in range(len(STAGES))] for i in range(len(STAGES))]))

mono = {}
for m in EXTRACTORS:
    E = emission(m)
    rho = []
    for k in range(n_c):
        r = np.argsort(np.argsort(E[:, k]))
        rho.append(float(np.corrcoef(r, np.arange(n_st))[0, 1]))
    mono[m] = rho


def kappa(a, b):
    a, b = np.array(a), np.array(b)
    po = np.mean(a == b)
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float((po - pe) / (1 - pe)) if pe < 1 else 1.0


kappas = {}
for i in range(len(EXTRACTORS)):
    for j in range(i + 1, len(EXTRACTORS)):
        a, b = EXTRACTORS[i], EXTRACTORS[j]
        kappas[f"{a} vs {b}"] = [kappa([c["cues"][a][k] for c in corpus], [c["cues"][b][k] for c in corpus])
                                 for k in range(n_c)]


def nb_accuracy(train_model, test_model):
    E = emission(train_model)
    prior = np.full(n_st, 1 / n_st)
    hits = 0
    for c in corpus:
        o = np.array(c["cues"][test_model])
        lik = np.prod(np.where(o[None, :] == 1, E, 1 - E), axis=1) * prior
        hits += int(lik.argmax() == c["stage"])
    return hits / len(corpus)


info = {m: nb_accuracy(m, HELD_OUT) for m in EXTRACTORS}
info["same_extractor_upper_bound"] = {m: nb_accuracy(m, m) for m in EXTRACTORS}

checks = {
    "stage_recoverability": {"judge": JUDGE, "accuracy": acc, "within_one_level": within_one,
                             "n": len(judged), "unparsed": n_judge_fail, "chance": 1 / n_st,
                             "chance_within_one": chance_within_one},
    "unparsed_extractions": {m: PARSE_FAIL.get(m, 0) for m in EXTRACTORS + [HELD_OUT]},
    "cue_monotonicity_spearman": mono,
    "extractor_agreement_kappa": kappas,
    "stage_from_cues_accuracy_on_held_out": info,
    "chance": 1 / n_st,
}
json.dump({"stages": STAGES, "cues": [c[0] for c in CUES], "profiles": list(PROFILES),
           "extractors": EXTRACTORS, "held_out": HELD_OUT, "judge": JUDGE, "emissions": emissions},
          open(f"{OUT_DIR}/emission.json", "w"), indent=1)
json.dump(checks, open(f"{OUT_DIR}/checks.json", "w"), indent=1)

print(json.dumps(checks, indent=1))
print(f"done in {time.time() - t0:.0f} s, {len(corpus)} messages, files in {os.path.abspath(OUT_DIR)}")


import os
import time
import json
import numpy as np
from itertools import product

QUICK = QUICK
CORPUS_DIR = "llm_corpus"
OUT_DIR = "results"
SEED = 0

rng = np.random.default_rng(SEED)

M_STAGES, BAD, N_CUES = 5, 4, 4
STRATEGIES = [
    ("reinforce companionship", +2.0, 1.00),
    ("validate neutrally", +1.0, 0.80),
    ("stay task oriented", 0.0, 0.60),
    ("challenge claims", -1.0, 0.45),
    ("redirect to humans", -2.0, 0.25),
]
A, FALLBACK = len(STRATEGIES), len(STRATEGIES) - 1
PRESSURE = np.array([s[1] for s in STRATEGIES])
UTIL = np.array([s[2] for s in STRATEGIES])

ESCALATION_RATIO = 7.4
SYCOPHANCY_RATE = 0.70
INFLUENCE_PERSISTENCE = 8
P_BASE = 0.06
GAMMA = np.log(ESCALATION_RATIO) / 2.0
RECOVERY_BASE = 1.0 / INFLUENCE_PERSISTENCE

BETA = 1.0
DELTA, K_PLAN, L_SAFE, N_SAMPLES = 0.10, 2, 8, 64
T_EPISODE = 30
N_EPISODES = 8 if QUICK else 60
GFN_ITERS = 400 if QUICK else 6000

PROFILES = {"low": (0.8, 1.2, 1.3), "medium": (1.1, 0.9, 1.1), "high": (1.6, 0.7, 0.9)}
PROFILE_VEC = {k: np.array(v) for k, v in PROFILES.items()}


def transition_matrix(pressure, susceptibility, recovery):
    up = np.clip(P_BASE * susceptibility * np.exp(GAMMA * pressure), 0.0, 0.9)
    dn = np.clip(RECOVERY_BASE * recovery * np.exp(-GAMMA * pressure), 0.0, 0.6)
    P = np.zeros((M_STAGES, M_STAGES))
    for z in range(M_STAGES):
        row = np.zeros(M_STAGES)
        row[z] = max(1e-6, 1 - up - dn)
        if z + 1 < M_STAGES:
            row[z + 1] = up
        else:
            row[z] += up
        if z - 1 >= 0:
            row[z - 1] = dn
        else:
            row[z] += dn
        P[z] = row / row.sum()
    return P


def fit_logistic(z, k, n, quadratic):
    X = np.stack([np.ones_like(z), z] + ([z ** 2] if quadratic else []), 1)
    w = np.zeros(X.shape[1])
    for _ in range(100):
        p = 1 / (1 + np.exp(-(X @ w)))
        g = X.T @ (k - n * p)
        H = (X * (n * p * (1 - p))[:, None]).T @ X + 1e-6 * np.eye(len(w))
        step = np.linalg.solve(H, g)
        w += step
        if np.abs(step).max() < 1e-8:
            break
    p = np.clip(1 / (1 + np.exp(-(X @ w))), 1e-9, 1 - 1e-9)
    ll = float(np.sum(k * np.log(p) + (n - k) * np.log(1 - p)))
    return w, ll


def fit_irt(extractor, profile):
    z = np.arange(M_STAGES, dtype=float)
    out = []
    for c in range(N_CUES):
        k = np.array([sum(r[c] for r in CUE_ROWS[(extractor, profile, s)]) for s in range(M_STAGES)], float)
        n = np.array([len(CUE_ROWS[(extractor, profile, s)]) for s in range(M_STAGES)], float)
        w1, ll1 = fit_logistic(z, k, n, False)
        w2, ll2 = fit_logistic(z, k, n, True)
        quad = 2 * (ll2 - ll1) > 3.84
        w = w2 if quad else w1
        X = np.stack([np.ones_like(z), z] + ([z ** 2] if quad else []), 1)
        fitted = 1 / (1 + np.exp(-(X @ w)))
        entry = {"form": "unimodal" if quad else "monotone", "coef": w.tolist(),
                 "fitted": fitted.tolist(), "empirical": (k / n).tolist(),
                 "max_abs_error": float(np.abs(fitted - k / n).max())}
        if not quad:
            entry["discrimination"] = float(w[1])
            entry["difficulty"] = float(-w[0] / w[1]) if abs(w[1]) > 1e-9 else float("nan")
        out.append(entry)
    E = np.clip(np.array([e["fitted"] for e in out]).T, 0.03, 0.97)
    return E, out


META = json.load(open(f"{CORPUS_DIR}/emission.json"))
EXTRACTORS, HELD_OUT = META["extractors"], META["held_out"]
CUE_ROWS = {}
with open(f"{CORPUS_DIR}/corpus.jsonl") as f:
    for line in f:
        c = json.loads(line)
        for m, cues in c["cues"].items():
            CUE_ROWS.setdefault((m, c["profile"], c["stage"]), []).append(cues)

IRT_E, IRT_REPORT = {}, {}
for m in EXTRACTORS + [HELD_OUT]:
    for p in PROFILES:
        IRT_E[(m, p)], IRT_REPORT[(m, p)] = fit_irt(m, p)


def make_user_model(profile, E):
    susceptibility, recovery, _ = profile
    P = np.stack([transition_matrix(PRESSURE[a], susceptibility, recovery) for a in range(A)])
    return {"P": P, "E": E, "profile": np.array(profile)}


def uncertainty_set(key, M=3, spread=0.25, seed=0):
    r = np.random.default_rng(seed)
    out = []
    for m in range(M):
        pert = np.array(PROFILES[key]) * (1 + spread * r.uniform(-1, 1, 3))
        out.append(make_user_model(tuple(pert), IRT_E[(EXTRACTORS[m % len(EXTRACTORS)], key)]))
    return out


def emit(model, z, r):
    return (r.random(N_CUES) < model["E"][z]).astype(int)


def belief_update(model, b, a, o):
    E = model["E"]
    lik = np.prod(np.where(o[None, :] == 1, E, 1 - E), axis=1)
    post = (b @ model["P"][a]) * lik
    s = post.sum()
    return post / s if s > 1e-12 else np.ones(M_STAGES) / M_STAGES


def executed(plan, L):
    return list(plan) + [FALLBACK] * (L - len(plan))


def plan_safe_prob(model, plan, L):
    f = np.ones(M_STAGES)
    f[BAD] = 0.0
    for a in reversed(executed(plan, L)):
        f = model["P"][a] @ f
        f[BAD] = 0.0
    return f


def plan_utility(plan):
    return float(sum(UTIL[a] for a in plan))


class SafetyTable:
    def __init__(self, ensemble, k=K_PLAN, L=L_SAFE):
        self.plans = list(product(range(A), repeat=k))
        self.idx = {p: i for i, p in enumerate(self.plans)}
        self.k, self.L, self.ens = k, L, ensemble
        self.F = [np.stack([plan_safe_prob(m, p, L) for p in self.plans]) for m in ensemble]

    def value(self, beliefs, i, robust=True):
        if robust:
            return min(float(beliefs[m] @ self.F[m][i]) for m in range(len(self.F)))
        return float(beliefs[0] @ self.F[0][i])

    def admissible(self, beliefs, delta, robust=True):
        ok = [i for i in range(len(self.plans)) if self.value(beliefs, i, robust) >= 1 - delta]
        return ok, sorted({self.plans[i][0] for i in ok})

    def reward(self, beliefs, b_nom, delta, i, robust=True):
        if self.value(beliefs, i, robust) < 1 - delta:
            return 0.0
        return float(b_nom @ self.F[0][i]) * np.exp(BETA * plan_utility(self.plans[i]))


class MLP:
    def __init__(self, sizes, seed=0, lr=3e-3):
        r = np.random.default_rng(seed)
        self.W = [r.standard_normal((a, b)) * np.sqrt(2 / a) for a, b in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(b) for b in sizes[1:]]
        self.lr, self.t = lr, 0
        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(x) for x in self.b]
        self.vb = [np.zeros_like(x) for x in self.b]

    def forward(self, x):
        self.h = [x]
        for i, (w, b) in enumerate(zip(self.W, self.b)):
            z = self.h[-1] @ w + b
            self.h.append(np.tanh(z) if i < len(self.W) - 1 else z)
        return self.h[-1]

    def backward(self, dout):
        gW, gb, d = [None] * len(self.W), [None] * len(self.W), dout
        for i in reversed(range(len(self.W))):
            gW[i], gb[i] = self.h[i].T @ d, d.sum(0)
            if i > 0:
                d = (d @ self.W[i].T) * (1 - self.h[i] ** 2)
        self.t += 1
        for i in range(len(self.W)):
            for p, g, m, v in ((self.W, gW, self.mW, self.vW), (self.b, gb, self.mb, self.vb)):
                m[i] = 0.9 * m[i] + 0.1 * g[i]
                v[i] = 0.999 * v[i] + 0.001 * g[i] ** 2
                p[i] -= self.lr * (m[i] / (1 - 0.9 ** self.t)) / (np.sqrt(v[i] / (1 - 0.999 ** self.t)) + 1e-8)


def log_softmax_masked(logits, mask):
    z = np.where(mask, logits, -1e9)
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def sample_training_belief():
    z = rng.choice(M_STAGES, p=[0.35, 0.30, 0.20, 0.10, 0.05])
    b = np.full(M_STAGES, 0.05) + np.eye(M_STAGES)[z] * rng.uniform(0.5, 3.0)
    b = b + rng.dirichlet(np.ones(M_STAGES) * 0.5) * 0.3
    return b / b.sum()


class GFlowNet:
    def __init__(self, k=K_PLAN, seed=1, lr=5e-4):
        self.k = k
        self.dim = M_STAGES + 3 + k + k * A
        self.net = MLP([self.dim, 64, 64, A], seed=seed, lr=lr)
        self.zn = MLP([M_STAGES + 3, 32, 1], seed=seed + 7, lr=1e-3)

    def log_z(self, b, xu):
        return float(self.zn.forward(np.concatenate([b, xu])[None])[0, 0])

    def feats(self, b, xu, step, partial):
        x = np.zeros(self.dim)
        x[:M_STAGES] = b
        x[M_STAGES:M_STAGES + 3] = xu
        x[M_STAGES + 3 + step] = 1.0
        for j, a in enumerate(partial):
            x[M_STAGES + 3 + self.k + j * A + a] = 1.0
        return x

    @staticmethod
    def allowed_at(ok_plans, tb, prefix):
        j = len(prefix)
        return sorted({tb.plans[i][j] for i in ok_plans if tb.plans[i][:j] == tuple(prefix)})

    def sample(self, b, xu, n, tb, ok_plans):
        plans = []
        for _ in range(n):
            partial = []
            for step in range(self.k):
                al = self.allowed_at(ok_plans, tb, partial) or list(range(A))
                mask = np.zeros((1, A), bool)
                mask[0, al] = True
                ls = log_softmax_masked(self.net.forward(self.feats(b, xu, step, partial)[None]), mask)[0]
                partial.append(int(rng.choice(A, p=np.exp(ls))))
            plans.append(tuple(partial))
        return plans

    def train(self, tables, delta, iters=GFN_ITERS, batch=32, robust=True):
        keys = list(tables)
        losses = []
        for _ in range(iters):
            tb, xu = tables[keys[rng.integers(len(keys))]]
            b = sample_training_belief()
            beliefs = [b] * len(tb.F)
            ok, allowed = tb.admissible(beliefs, delta, robust)
            if not allowed:
                continue
            plans = self.sample(b, xu, batch, tb, ok)
            logR = np.log([max(tb.reward(beliefs, b, delta, tb.idx[p], robust), 1e-12) for p in plans])
            X = np.array([self.feats(b, xu, s, p[:s]) for p in plans for s in range(self.k)])
            ACT = np.array([p[s] for p in plans for s in range(self.k)])
            mask = np.zeros((len(X), A), bool)
            for bi, p in enumerate(plans):
                for s in range(self.k):
                    for a in self.allowed_at(ok, tb, list(p[:s])):
                        mask[bi * self.k + s, a] = True
            ls = log_softmax_masked(self.net.forward(X), mask)
            logpf = ls[np.arange(len(X)), ACT].reshape(batch, self.k).sum(1)
            cond = np.concatenate([b, xu])[None]
            lz = self.zn.forward(cond)[0, 0]
            resid = lz + logpf - logR
            losses.append(float((resid ** 2).mean()))
            d = 2 * resid / batch
            dout = -np.exp(ls) * np.repeat(d, self.k)[:, None]
            dout[np.arange(len(X)), ACT] += np.repeat(d, self.k)
            self.net.backward(dout)
            self.zn.forward(cond)
            self.zn.backward(np.array([[float(d.sum())]]))
        return losses


def pol_helpful(**kw):
    return 0, False


def pol_deployed(**kw):
    if rng.random() < SYCOPHANCY_RATE:
        return int(rng.integers(0, 2)), False
    return int(rng.integers(2, A)), False


TURN_TAU = [None]


def pol_turnlevel(table, delta, obs, **kw):
    E = table.ens[0]["E"]
    if obs is None:
        b_cur = np.eye(M_STAGES)[0]
    else:
        lik = np.prod(np.where(obs[None, :] == 1, E, 1 - E), axis=1)
        b_cur = lik / lik.sum()
    risk = [float((b_cur @ table.ens[0]["P"][a])[BAD]) for a in range(A)]
    tau = TURN_TAU[0] if TURN_TAU[0] is not None else delta / table.L
    ok = [a for a in range(A) if risk[a] <= tau]
    return (max(ok, key=lambda a: UTIL[a]) if ok else FALLBACK), False


def pol_reactive(b_nom, **kw):
    return (FALLBACK if b_nom[3:].sum() > 0.35 else 0), False


def pol_cra(state, b_nom, **kw):
    s = state.setdefault("cra", {"hist": [], "b0": b_nom.copy()})
    drift = 1 - float(b_nom @ s["b0"]) / (np.linalg.norm(b_nom) * np.linalg.norm(s["b0"]) + 1e-9)
    s["hist"].append(float(b_nom[3:].sum()))
    w = s["hist"][-6:]
    slope = float(np.polyfit(range(len(w)), w, 1)[0]) if len(w) > 2 else 0.0
    score = 0.35 * drift + 0.45 * float(b_nom[3:].sum()) + 0.20 * max(slope, 0.0) * 6
    return (FALLBACK if score > 0.45 else 0), False


def pol_shield(table, beliefs, delta, robust=True, **kw):
    _, acts = table.admissible(beliefs, delta, robust)
    return (max(acts, key=lambda a: UTIL[a]) if acts else FALLBACK), (not acts)


def pol_proposed(table, beliefs, b_nom, xu, gfn, delta, n_samples=N_SAMPLES, robust=True, **kw):
    ok, acts = table.admissible(beliefs, delta, robust)
    if not acts:
        return FALLBACK, True
    counts = np.zeros(A)
    for p in gfn.sample(b_nom, xu, n_samples, table, ok):
        counts[p[0]] += 1
    return int(counts.argmax()), False


POLICIES = {
    "always helpful": pol_helpful,
    "deployed-like (70% sycophantic)": pol_deployed,
    "turn-level filter (current turn)": pol_turnlevel,
    "crisis detection": pol_reactive,
    "CRA accumulation": pol_cra,
    "shield + myopic": pol_shield,
    "proposed": pol_proposed,
}
USES_ADMISSIBLE = {"shield + myopic", "proposed"}


def run_episode(policy, user, table, gfn, xu, delta, T=T_EPISODE, seed=0, n_samples=N_SAMPLES, robust=True):
    r = np.random.default_rng(seed)
    z = 0
    beliefs = [np.eye(M_STAGES)[0].copy() for _ in table.F]
    state, util, fb, empty = {}, 0.0, 0, 0
    harmed = False
    o = None
    harm_t, stage_t = np.zeros(T), np.zeros(T)
    for t in range(T):
        b_nom = np.mean(beliefs, axis=0)
        a, was_empty = policy(table=table, beliefs=beliefs, b_nom=b_nom, xu=xu, gfn=gfn,
                              delta=delta, n_samples=n_samples, robust=robust, state=state, obs=o)
        fb += (a == FALLBACK)
        empty += was_empty
        util += UTIL[a]
        z = r.choice(M_STAGES, p=user["P"][a, z])
        harmed = harmed or (z == BAD)
        harm_t[t], stage_t[t] = harmed, z
        o = emit(user, z, r)
        beliefs = [belief_update(m, b, a, o) for m, b in zip(table.ens, beliefs)]
    return dict(harmful=harmed, utility=util / T, fallback=fb / T, empty=empty / T,
                harm_curve=harm_t, stage_curve=stage_t)


def evaluate(policy, users, tables, gfn, delta, n_ep=N_EPISODES, n_samples=N_SAMPLES, robust=True, seed_offset=0):
    res = []
    for ui, (key, user) in enumerate(users.items()):
        tb, xu = tables[key]
        for ep in range(n_ep):
            res.append(run_episode(policy, user, tb, gfn, xu, delta, seed=seed_offset + 1000 * ui + ep,
                                   n_samples=n_samples, robust=robust))
    return res


def summarize(res):
    return dict(harmful=float(np.mean([r["harmful"] for r in res])),
                utility=float(np.mean([r["utility"] for r in res])),
                fallback=float(np.mean([r["fallback"] for r in res])),
                empty=float(np.mean([r["empty"] for r in res])),
                harm_curve=np.mean([r["harm_curve"] for r in res], axis=0),
                stage_curve=np.mean([r["stage_curve"] for r in res], axis=0))


def build(delta=DELTA, L=L_SAFE, k=K_PLAN, M=3, iters=GFN_ITERS):
    users = {key: make_user_model(v, IRT_E[(HELD_OUT, key)]) for key, v in PROFILES.items()}
    tables = {key: (SafetyTable(uncertainty_set(key, M=M, seed=i + 1), k=k, L=L), PROFILE_VEC[key])
              for i, key in enumerate(PROFILES)}
    g = GFlowNet(k=k, lr=5e-4)
    if iters > 0:
        g.train(tables, delta, iters=iters)
    return users, tables, g


PARETO_DELTAS = (0.02, 0.05, 0.10, 0.20, 0.40)
N_EP_PARETO = 8 if QUICK else 40


def calibrate_turn_level(users, tables, g, delta):
    ref = summarize(evaluate(pol_proposed, users, tables, g, delta, n_ep=4 if QUICK else 20, seed_offset=500000))
    n_cal = 4 if QUICK else 20
    best = None
    for tau in np.geomspace(1e-3, 0.5, 16):
        TURN_TAU[0] = float(tau)
        u = summarize(evaluate(pol_turnlevel, users, tables, g, delta, n_ep=n_cal, seed_offset=500000))["utility"]
        if best is None or abs(u - ref["utility"]) < best[1]:
            best = (float(tau), abs(u - ref["utility"]))
    TURN_TAU[0] = best[0]


def exact_plan_distribution(tb, beliefs, b, delta):
    r = np.array([tb.reward(beliefs, b, delta, i, True) for i in range(len(tb.plans))])
    return r / r.sum() if r.sum() > 0 else r


def modes_data(tables, g, delta):
    local = np.random.default_rng(23)
    cases, cov_g, cov_a = [], [], []
    tries = 0
    while len(cases) < 3 and tries < 5000:
        tries += 1
        key = list(tables)[local.integers(len(tables))]
        tb, xu = tables[key]
        b = sample_training_belief()
        beliefs = [b] * len(tb.F)
        ok, acts = tb.admissible(beliefs, delta, True)
        if len(acts) < 3:
            continue
        ex = exact_plan_distribution(tb, beliefs, b, delta)
        if ex.sum() <= 0 or int((ex > 0.5 * ex.max()).sum()) < 3:
            continue
        cnt = np.zeros(len(tb.plans))
        for p in g.sample(b, xu, 500 if QUICK else 3000, tb, ok):
            cnt[tb.idx[p]] += 1
        cases.append({"profile": key, "plans": [list(p) for p in tb.plans], "exact": ex.tolist(),
                      "sampled": (cnt / cnt.sum()).tolist(), "n_modes": int((ex > 0.5 * ex.max()).sum())})
    n_cov = 0
    while n_cov < (10 if QUICK else 35):
        key = list(tables)[local.integers(len(tables))]
        tb, xu = tables[key]
        b = sample_training_belief()
        beliefs = [b] * len(tb.F)
        ok, acts = tb.admissible(beliefs, delta, True)
        if len(acts) < 2:
            continue
        ex = exact_plan_distribution(tb, beliefs, b, delta)
        if ex.sum() <= 0:
            continue
        modes = np.where(ex > 0.5 * ex.max())[0]
        cnt = np.zeros(len(tb.plans))
        for p in g.sample(b, xu, N_SAMPLES, tb, ok):
            cnt[tb.idx[p]] += 1
        cov_g.append(float(np.mean(cnt[modes] > 0)))
        cov_a.append(1.0 / len(modes))
        n_cov += 1
    return {"cases": cases, "coverage": {"gflownet": float(np.mean(cov_g)), "argmax": float(np.mean(cov_a))}}


os.makedirs(OUT_DIR, exist_ok=True)
t0 = time.time()
users, tables, g = build()
calibrate_turn_level(users, tables, g, DELTA)
print(f"turn-level threshold: tau = {TURN_TAU[0]:.4f}")

data = {"main": {}}
for name, pol in POLICIES.items():
    s_ = summarize(evaluate(pol, users, tables, g, DELTA))
    data["main"][name] = {"harm": 100 * s_["harmful"], "utility": s_["utility"],
                          "harm_curve": (100 * s_["harm_curve"]).tolist(),
                          "stage_curve": s_["stage_curve"].tolist()}
    print(f"{name:<34}{100 * s_['harmful']:>7.1f}%{s_['utility']:>7.2f}")

data["pareto"] = []
for d_ in PARETO_DELTAS:
    _, tb_, g_ = build(delta=d_)
    s_ = summarize(evaluate(pol_proposed, users, tb_, g_, d_, n_ep=N_EP_PARETO))
    data["pareto"].append({"delta": d_, "harm": 100 * s_["harmful"], "utility": s_["utility"]})
    print(f"pareto delta={d_:g}: harm {100 * s_['harmful']:.1f}%, helpfulness {s_['utility']:.2f}")

data["modes"] = modes_data(tables, g, DELTA)
print(f"modes covered: {100 * data['modes']['coverage']['gflownet']:.0f}% "
      f"(single-solution planner: {100 * data['modes']['coverage']['argmax']:.0f}%)")

json.dump(data, open(f"{OUT_DIR}/results.json", "w"), indent=1, default=float)
print(f"done in {time.time() - t0:.0f} s, results saved in {os.path.abspath(OUT_DIR)}/results.json")
